This notebook trains the two models used by the rice detector app. It is the same pipeline as rice_pipeline.ipynb, with checks added around the parts that are easy to get wrong.

Step 1 trains the strain classifier, which reads the rice variety from a photo of grains.

Step 2 trains the lesion detector, which finds disease on a leaf and counts how many lesions of each type are present.

The difference from the original notebook is that this one verifies what it assumes. It confirms the GPU is really enabled, finds the image folders instead of trusting a hardcoded name, locates and repairs the data.yaml file for the detection dataset, and records the library versions so the server can match them later.

Run the cells in order from the top. Expect three to five hours on a T4.

Check the GPU before anything else.

The cell below lists the GPU that Colab has attached. If it prints nothing useful, the runtime has no GPU, and training would take days instead of hours. Go to Runtime, then Change runtime type, and select T4 GPU.

There is a second check after the install that stops the notebook outright if no GPU is available. It runs after the install rather than before it, because importing torch before pip has finished can leave the session holding a half replaced library.

In [ ]:
!nvidia-smi

Install the libraries needed for both steps, then confirm the GPU is really there.

If the import cell further down fails after this, Colab has swapped a torch version underneath the install. Use Runtime, then Restart session, and continue from the import cell. Do not run the install again.

In [ ]:
!pip install fastai timm ultralytics opencv-python scikit-learn kagglehub pyyaml -q

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU. Runtime -> Change runtime type -> T4 GPU, then re-run."
)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
from fastai.vision.all import *
from ultralytics import YOLO
import kagglehub

# stdlib last: a star import can shadow these names
from pathlib import Path
from collections import Counter
import json, shutil, yaml

# Everything the run produces lands here, then gets zipped at the end.
OUT = Path("/content/artifacts")
OUT.mkdir(parents=True, exist_ok=True)
print("fastai/torch imports OK")

Kaggle credentials.

Both datasets are downloaded from Kaggle, which requires an API key. Without one the download fails with a 401 that reads like a network problem.

In the Colab sidebar, open Secrets and add KAGGLE_USERNAME and KAGGLE_KEY from your Kaggle account settings. Enable notebook access for both. The names have to match exactly.

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("Kaggle credentials loaded from Colab secrets.")
except Exception as e:
    print("Could not load Colab secrets:", e)
    print("Falling back to any kaggle.json already present. If downloads 401, fix this first.")

Step 1: Strain Classification

The dataset is the Rice Image Dataset from Kaggle. It contains five rice varieties, Arborio, Basmati, Ipsala, Jasmine, and Karacadag.

Rice Image Dataset
https://www.kaggle.com/datasets/muratkokludataset/rice-image-dataset

The images are individual rice grains photographed on a plain background, not leaves and not field photos. The model that comes out of this step has never seen a leaf. Given one it does not return low confidence, it returns a confident wrong variety, because the five probabilities have to add up to one whatever the input is. This is why the app asks for a separate grain photo and reports the strain as not assessed when that photo is skipped.

In [ ]:
strain_dataset_path = kagglehub.dataset_download("muratkokludataset/rice-image-dataset")
print("Downloaded to", strain_dataset_path)

Find the folder that holds the variety subfolders.

The original notebook hardcodes Rice_Image_Dataset as the subfolder name. Neither way of getting this wrong is loud. If the folder is missing, the image list comes back empty and the error surfaces later, somewhere else. If the folder exists but sits at the wrong level, the run is silently corrupt, because the label for each image is taken from the name of its parent folder.

The cell below looks for the directory whose subfolders actually contain images, and prints the per class file counts so they can be checked.

In [ ]:
def find_class_root(root: Path, min_classes: int = 2) -> Path:
    """Deepest directory whose subfolders directly contain images."""
    best, best_count = None, 0
    for d in [root, *root.rglob("*")]:
        if not d.is_dir():
            continue
        subs = [s for s in d.iterdir() if s.is_dir()]
        if len(subs) < min_classes:
            continue
        # a class folder holds images directly
        ok = [s for s in subs if any(s.glob("*.jpg")) or any(s.glob("*.png"))]
        if len(ok) >= min_classes and len(ok) > best_count:
            best, best_count = d, len(ok)
    if best is None:
        raise RuntimeError(f"No class-folder layout found under {root}")
    return best


STRAIN_IMAGE_DIR = find_class_root(Path(strain_dataset_path))
classes = sorted(d.name for d in STRAIN_IMAGE_DIR.iterdir() if d.is_dir())

print("Image root:", STRAIN_IMAGE_DIR)
print("Classes:", classes)
for c in classes:
    n = len(list((STRAIN_IMAGE_DIR / c).glob("*")))
    print(f"  {c}: {n} files")
print("Total images:", len(get_image_files(STRAIN_IMAGE_DIR)))

Configuration. These values are unchanged from the original notebook.

Training happens twice, once at 224 pixels and once at 358. Training small first and then larger is called progressive resizing, and it gives better accuracy than training at one fixed size.

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 64
ARCH = "convnext_small_in22k"
EPOCHS_HEAD = 3
EPOCHS_FINE_TUNE = 8
MODEL_OUT = OUT / "strain_classifier.pkl"


def build_dataloaders(img_size, bs):
    dblock = DataBlock(
        blocks=(ImageBlock, CategoryBlock),
        get_items=get_image_files,
        get_y=parent_label,
        splitter=RandomSplitter(valid_pct=0.2, seed=42),
        item_tfms=Resize(img_size, method="squish"),
        batch_tfms=aug_transforms(size=img_size, min_scale=0.75)
                   + [Normalize.from_stats(*imagenet_stats)],
    )
    return dblock.dataloaders(STRAIN_IMAGE_DIR, bs=bs)

Train on small images first. This trains quickly and confirms the pipeline works before spending time on the larger image size.

If the GPU runs out of memory here, halve BATCH_SIZE and run the cell again.

In [ ]:
dls = build_dataloaders(IMG_SIZE, BATCH_SIZE)
learn = vision_learner(dls, ARCH, metrics=error_rate).to_fp16()
learn.fine_tune(EPOCHS_HEAD, base_lr=3e-3)

Train again on larger images, at 358 pixels with the batch size halved to fit. This is the longest cell in the notebook.

In [ ]:
dls_big = build_dataloaders(int(IMG_SIZE * 1.6), BATCH_SIZE // 2)
learn.dls = dls_big
learn.fine_tune(EPOCHS_FINE_TUNE, base_lr=1e-3)

Save the trained model and print the vocabulary.

The order of the vocabulary matters. The predicted index points into it, and the API returns these strings to the app exactly as they appear here. Whatever this cell prints is what the class names actually are.

In [ ]:
learn.export(MODEL_OUT)

strain_vocab = list(map(str, learn.dls.vocab))
print("Saved:", MODEL_OUT, f"({MODEL_OUT.stat().st_size / 1e6:.1f} MB)")
print("Vocab (order matters):", strain_vocab)

Evaluation. This checks the classifier against the ground truth labels and shows the training curves and the confusion matrix.

Look at the confusion matrix, not only the accuracy number. A high overall accuracy can still hide one variety being read as another every time, and a single number will not show that.

In [ ]:
learn.recorder.plot_loss()

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix(figsize=(8, 8))

In [ ]:
preds, targs = learn.tta()
strain_tta_acc = float((preds.argmax(dim=1) == targs).float().mean())
print(f"Validation accuracy with TTA: {strain_tta_acc:.4f}")

This shows individual predictions next to the ground truth label, worst first, so the errors can be inspected directly. It is the quickest way to spot label noise, or two varieties the model cannot tell apart.

In [ ]:
interp.plot_top_losses(9, nrows=3)

Step 2: Localized Disease Detection

Strain classification only says which variety a plant is. It cannot report three Leaf Blast lesions detected, because that means locating each lesion rather than labeling the whole image. This step trains a YOLO model for that.

Rice Leaf Diseases, YOLO formatted
https://www.kaggle.com/datasets/yusufmurtaza01/rice-leaf-diseases

Rice Leaf Spot Disease Annotated Dataset
https://www.kaggle.com/datasets/hadiurrahmannabil/rice-leaf-spot-disease-annotated-dataset

Either dataset works as long as it is in YOLO format, meaning one label file per image and a data.yaml describing the splits.

In [ ]:
lesion_dataset_path = kagglehub.dataset_download("yusufmurtaza01/rice-leaf-diseases")
print("Downloaded to", lesion_dataset_path)

Find the real data.yaml, and check what is inside it.

The original notebook builds this path as the download folder plus data.yaml. Nothing guarantees a file is there, and often it sits a directory or two further down.

The contents are the more dangerous half. The train and val entries are frequently absolute paths from whoever built the dataset, or relative to a folder that does not survive extraction. Ultralytics will start training with a validation split that points at nothing, and report numbers that mean nothing.

The cells below find the file, try each plausible location for the splits, and confirm the folders actually contain images. Record the class names that get printed, because the API uses them as keys.

In [ ]:
def has_images(d: Path) -> bool:
    if not d.is_dir():
        return False
    return any(d.rglob("*.jpg")) or any(d.rglob("*.png")) or any(d.rglob("*.jpeg"))


def resolve_split(spec, yaml_dir: Path, declared_root):
    """Try each plausible base until one resolves to a directory holding images."""
    if spec is None:
        return None
    bases = [yaml_dir]
    if declared_root:
        bases += [Path(declared_root), yaml_dir / declared_root]
    bases += [yaml_dir.parent]

    cands = [Path(spec)] if Path(spec).is_absolute() else []
    cands += [b / spec for b in bases]

    for c in cands:
        # A split may point straight at an images dir, or at a parent holding images/.
        # Prefer the images/ dir: ultralytics derives label paths by swapping
        # "/images/" for "/labels/", so handing it the parent is fragile.
        for probe in (Path(c) / "images", Path(c)):
            if has_images(probe):
                return probe.resolve()
    return None


def pick_data_yaml(root: Path):
    """First yaml that looks like a dataset config, not just the first one alphabetically.
    Datasets sometimes ship args.yaml or similar next to the real data.yaml."""
    for p in sorted(root.rglob("*.yaml")):
        try:
            c = yaml.safe_load(open(p))
        except Exception:
            continue
        if isinstance(c, dict) and "names" in c and ("train" in c or "val" in c):
            return p, c
    return None, None


yaml_candidates = sorted(Path(lesion_dataset_path).rglob("*.yaml"))
print("YAML files found:", [str(p) for p in yaml_candidates])
assert yaml_candidates, "No .yaml in the download - dataset is not in YOLO format."

data_yaml_path, cfg = pick_data_yaml(Path(lesion_dataset_path))
assert cfg is not None, "No .yaml looked like a dataset config (needs names, plus train or val)."
yaml_dir = data_yaml_path.parent
print("\nUsing:", data_yaml_path)
print(json.dumps(cfg, indent=2, default=str))

In [ ]:
declared_root = cfg.get("path")
resolved = {}

for split in ("train", "val", "test"):
    if split not in cfg:
        continue
    r = resolve_split(cfg[split], yaml_dir, declared_root)
    resolved[split] = r
    n = len([p for p in r.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]) if r else 0
    print(f"{split:5} -> {r}   ({n} images)")

assert resolved.get("train"), "train split did not resolve to any directory with images"
if not resolved.get("val"):
    print("\nWARNING: no val split resolved. It is dropped from the corrected yaml below, so")
    print("training stops with a clear error instead of reporting meaningless numbers.")

Write a corrected copy of the yaml, with absolute paths that have been checked, and use that for training. The original file is left alone.

In [ ]:
fixed = dict(cfg)
fixed.pop("path", None)          # absolute splits, so a root would only confuse things
for split in ("train", "val", "test"):
    if split not in cfg:
        continue
    r = resolved.get(split)
    if r:
        fixed[split] = str(r)
    else:
        # drop it rather than leave a path we already know is broken: ultralytics would
        # happily train against it and report numbers that mean nothing
        fixed.pop(split, None)

FIXED_YAML = OUT / "data_fixed.yaml"
yaml.safe_dump(fixed, open(FIXED_YAML, "w"), sort_keys=False)

print("Wrote", FIXED_YAML)
print(json.dumps(fixed, indent=2, default=str))
print("\nDisease classes ->", fixed.get("names"))

Train the detector.

The patience setting stops training early once the validation score stops improving, so finishing before epoch 100 is normal and not a failure.

Check the image counts in the first few lines of output against the counts printed above. If they disagree, the splits are not pointing where you think they are.

In [ ]:
model = YOLO("yolo11n.pt")

model.train(
    data=str(FIXED_YAML),
    epochs=100,
    imgsz=640,
    patience=15,
    name="rice_lesion_detector",
)

In [ ]:
metrics = model.val()
lesion_map50 = float(metrics.box.map50)
lesion_map = float(metrics.box.map)
print(f"mAP50:    {lesion_map50:.4f}")
print(f"mAP50-95: {lesion_map:.4f}")

The training run saves graphs and a confusion matrix automatically. These show the loss curves, precision, recall, and mAP across all epochs.

The pair of validation images is the most useful thing here. One has the ground truth boxes drawn on it and the other has the predicted boxes, on the same photos. Comparing them says more about whether the model works than any single number does.

In [ ]:
from PIL import Image as PILImage

results_dir = Path(model.trainer.save_dir)
print("Run directory:", results_dir)

for name in ["results.png", "confusion_matrix.png",
             "val_batch0_labels.jpg", "val_batch0_pred.jpg"]:
    p = results_dir / name
    if p.exists():
        print("\n" + name)
        display(PILImage.open(p))
    else:
        print("missing:", name)

Copy the weights and the training graphs into the artifacts folder, so everything can be downloaded together at the end.

In [ ]:
best_pt = results_dir / "weights" / "best.pt"
assert best_pt.exists(), f"best.pt not found at {best_pt}"

shutil.copy(best_pt, OUT / "best.pt")
shutil.copytree(results_dir, OUT / "yolo_run", dirs_exist_ok=True)

# names from the trained model are authoritative - the yaml is only what we asked for
lesion_names_final = model.names
print("Copied best.pt", f"({(OUT / 'best.pt').stat().st_size / 1e6:.1f} MB)")
print("Detector classes:", lesion_names_final)

Step 3: Collect the Results

Training is not finished when the last cell stops running. The server needs the library versions, and the class names and accuracy numbers need to be written down somewhere permanent.

Record the library versions.

The saved classifier is a Python pickle. It holds references to fastai, torch and timm classes by module path, so it only loads under matching versions. A server with different versions fails on load with an AttributeError or a ModuleNotFoundError that gives no hint the real problem is a version difference.

Do this now, in the same session that produced the file. Once the runtime is recycled the versions are gone.

In [ ]:
!pip freeze > /content/artifacts/requirements-training.txt

import importlib.metadata as md_meta

KEY = ["fastai", "torch", "torchvision", "timm", "numpy", "pillow", "ultralytics",
       "opencv-python", "scipy"]
versions = {}
for pkg in KEY:
    try:
        versions[pkg] = md_meta.version(pkg)
    except Exception:
        versions[pkg] = "not installed"

print("Pin these in backend/requirements.txt:\n")
for k, v in versions.items():
    print(f"  {k}=={v}")

Print the accuracy numbers and the class names as one block.

Copy this somewhere permanent, alongside the weights. Model files whose accuracy and class names were never written down cannot be trusted or rebuilt later.

In [ ]:
import platform

facts = {
    "trained_on": platform.platform(),
    "gpu": torch.cuda.get_device_name(0),
    "strain": {
        "arch": ARCH,
        "vocab": strain_vocab,
        "tta_accuracy": round(strain_tta_acc, 4),
        "image_count": len(get_image_files(STRAIN_IMAGE_DIR)),
    },
    "lesion": {
        "arch": "yolo11n",
        "names": lesion_names_final,
        "map50": round(lesion_map50, 4),
        "map50_95": round(lesion_map, 4),
        "conf_threshold": 0.4,
    },
    "versions": versions,
}

json.dump(facts, open(OUT / "model_facts.json", "w"), indent=2)

print("=" * 70)
print("PASTE INTO docs/01-models.md")
print("=" * 70)
print()
print("Strain classifier")
print("  vocab (exact order):", strain_vocab)
print(f"  TTA validation accuracy: {strain_tta_acc:.4f}")
print()
print("Lesion detector")
print("  classes:", lesion_names_final)
print(f"  mAP50: {lesion_map50:.4f}")
print(f"  mAP50-95: {lesion_map:.4f}")
print()
print("Pinned training versions:")
for k, v in versions.items():
    print(f"  {k}=={v}")

Zip everything and download it.

Colab runtimes are temporary. Do this before closing the tab. Retraining takes hours, not minutes.

In [ ]:
shutil.make_archive("/content/rice_models", "zip", OUT)
size = Path("/content/rice_models.zip").stat().st_size / 1e6
print(f"rice_models.zip - {size:.1f} MB")

from google.colab import files
files.download("/content/rice_models.zip")

Unpack the zip on your own machine and put the two model files into the models folder of the project.

strain_classifier.pkl
best.pt

Keep a backup of the zip somewhere outside the project as well. The weight files are excluded from version control because they are large, so nothing else is protecting them.

Keep requirements-training.txt too. That is the version list the server has to match.